<a href="https://colab.research.google.com/github/lcandau/histopathology-clip-lab/blob/exp_10_dinov2_backbone/experiments/exp_10_dinov2_backbone/CLIP_DINOv2_BERT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# exp_10 — DINOv2 image encoder + BERT text encoder

Swap the image backbone from keras_hub's ImageNet-pretrained ResNet50 to **DINOv2's ViT-B/14 vision encoder** (`facebook/dinov2-base`, self-supervised on the general-domain LVD-142M image dataset). Keep the text side identical to exp_06's best biomed cell and to exp_08:

- HF `bert-base-uncased`, frozen
- Composed prompt: `"A histopathology image of {class_name}."`

Frozen-backbones + projection heads + InfoNCE, same as the rest of the CLIP-style notebooks. **Control experiment** for exp_08 (PLIP): same family (ViT, self-supervised) but general-domain pretraining instead of histopathology. See `README.md` for the design rationale.

## NOTE: DINOv2 features are precomputed in PyTorch

`transformers==4.46.0` has no TensorFlow port of DINOv2 (`TFAutoModel.from_pretrained('facebook/dinov2-base')` raises `ValueError: Unrecognized configuration class ... for TFAutoModel`). Since the vision backbone is frozen, we precompute the 768-d CLS features once with PyTorch (see `DINOv2_feature_precompute.ipynb`) and cache them to Drive. This notebook reads from that cache and only trains the projection heads + `logit_scale` in TF/Keras.

**Prereq**: run `DINOv2_feature_precompute.ipynb` first so `/content/drive/MyDrive/clip_histopathology/cache/dinov2/lc25000.h5` exists.

## 0 — Bootstrap

In [ ]:
# --- Cell 0: bootstrap ---
import os, sys, subprocess
os.environ["KERAS_BACKEND"] = "tensorflow"

REPO_URL = "https://github.com/lcandau/histopathology-clip-lab.git"
REPO_DIR = "/content/histopathology-clip-lab"
BRANCH   = "exp_10_dinov2_backbone"
IN_COLAB = "google.colab" in sys.modules


def _clone_with_fallback(branch):
    try:
        subprocess.run(
            ["git", "clone", "-b", branch, REPO_URL, REPO_DIR],
            check=True, capture_output=True,
        )
        print(f"Cloned branch {branch!r}")
        return branch
    except subprocess.CalledProcessError:
        print(f"Branch {branch!r} not found; cloning main")
        subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
        return "main"


if IN_COLAB:
    if not os.path.exists(REPO_DIR):
        active = _clone_with_fallback(BRANCH)
    else:
        fetch = subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", BRANCH], capture_output=True)
        if fetch.returncode == 0:
            subprocess.run(["git", "-C", REPO_DIR, "checkout", BRANCH], check=True)
            subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard", f"origin/{BRANCH}"], check=True)
            active = BRANCH
        else:
            subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", "main"], check=True)
            subprocess.run(["git", "-C", REPO_DIR, "checkout", "main"], check=True)
            subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard", "origin/main"], check=True)
            active = "main"
    print(f"Active branch: {active}")
    subprocess.run([
        "pip", "install", "-q",
        "tensorflow==2.18.0", "keras==3.7.0", "keras-hub==0.18.1",
        "transformers==4.46.0", "umap-learn", "kagglehub",
        "scikit-learn", "matplotlib", "pandas", "Pillow",
    ], check=True)
    subprocess.run(["pip", "uninstall", "-y", "-q", "jax", "jaxlib"], check=False)
    if REPO_DIR not in sys.path:
        sys.path.insert(0, REPO_DIR)
    from google.colab import drive
    if not os.path.exists("/content/drive/MyDrive"):
        drive.mount("/content/drive")
else:
    LOCAL_REPO_DIR = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
    if LOCAL_REPO_DIR not in sys.path:
        sys.path.insert(0, LOCAL_REPO_DIR)

print("In Colab:", IN_COLAB)
print("sys.path[0]:", sys.path[0])
print("KERAS_BACKEND:", os.environ.get("KERAS_BACKEND"))

## 1 — Imports

In [ ]:
# --- Cell 1: imports ---
import os
os.environ.setdefault("KERAS_BACKEND", "tensorflow")

import json
import math
import random
from pathlib import Path

import numpy as np
import pandas as pd
import h5py
import matplotlib.pyplot as plt
from PIL import Image

import tensorflow as tf
import keras

from transformers import AutoTokenizer, TFAutoModel
# NOTE: DINOv2 is *not* loaded here -- it has no TF port in transformers==4.46.0.
# We precompute features in PyTorch (DINOv2_feature_precompute.ipynb) and load
# them from an HDF5 cache below.

from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score,
    classification_report, confusion_matrix, f1_score,
)
import umap

from src.utils.paths import is_colab, repo_root, run_dir, results_dir
from src.utils.repro import set_global_seed, enable_op_determinism
from src.data.lc25000 import (
    CLASS_INFO, NUM_CLASSES, CLASS_NAMES, INDEX_TO_NAME,
    discover_records, stratified_split, save_split, load_split,
)
from src.prompts.templates import baseline_prompt

## 2 — Configuration

The DINOv2 image preprocessing (resize + ImageNet mean/std) happens **inside the precompute notebook** (`DINOv2_feature_precompute.ipynb`). At training time we only read 768-d feature vectors out of an HDF5 cache, so no per-step image decode + normalisation is needed here.

In [ ]:
# --- Cell 2: hyperparameters ---
EXPERIMENT = "exp_10_dinov2_backbone"
VERSION    = "dinov2_bert_composed"

SEED = 42
set_global_seed(SEED)
enable_op_determinism()

# Image input -- precompute used 224x224, CLS feature is 768-d.
IMG_SIZE       = 224
VISION_HIDDEN  = 768   # DINOv2 ViT-B/14 hidden size

# Text input.
MAX_LEN = 24  # matches exp_06 / exp_08 bert_composed cells

# Shared embedding space.
EMBED_DIM = 256

# Optimisation -- held identical to exp_06 / exp_08 to keep the comparison clean.
BATCH_SIZE = 32
EPOCHS = 30
EARLY_STOPPING_PATIENCE = 5
LR = 1e-3
WD = 1e-4
GLOBAL_CLIPNORM = 1.0
INIT_TEMP = 0.07

AUTOTUNE = tf.data.AUTOTUNE
FORCE_RETRAIN = False

DINOV2_MODEL_ID = "facebook/dinov2-base"
TEXT_ENCODER_ID = "bert-base-uncased"

# Precomputed feature cache (produced by DINOv2_feature_precompute.ipynb).
CACHE_ROOT = Path("/content/drive/MyDrive/clip_histopathology/cache/dinov2")
FEATURE_CACHE_PATH = CACHE_ROOT / "lc25000.h5"

# Output directories.
RUN_DIR = run_dir(EXPERIMENT, VERSION)
SPLIT_PATH = results_dir("splits") / f"lc25000_seed{SEED}.json"
METRICS_DIR = results_dir("metrics") / EXPERIMENT
PLOTS_DIR = results_dir("plots") / EXPERIMENT
CM_DIR = results_dir("confusion_matrices") / EXPERIMENT
for d in (METRICS_DIR, PLOTS_DIR, CM_DIR):
    d.mkdir(parents=True, exist_ok=True)

print(f"Run dir:        {RUN_DIR}")
print(f"DINOv2 model:   {DINOV2_MODEL_ID}")
print(f"Text model:     {TEXT_ENCODER_ID}")
print(f"Split path:     {SPLIT_PATH}")
print(f"Feature cache:  {FEATURE_CACHE_PATH}")
if not FEATURE_CACHE_PATH.exists():
    raise FileNotFoundError(
        f"\nPrecomputed DINOv2 features not found at {FEATURE_CACHE_PATH}.\n"
        f"Run 'DINOv2_feature_precompute.ipynb' first to build the cache, then\n"
        f"re-run this notebook.\n"
    )

## 3 — Dataset

Same LC25000 split as the rest of the project (loaded from the canonical pre-existing JSON on Drive). Pre-dedupe split — `apply_dedupe=True` in `discover_records` is only triggered if the JSON does not already exist, which it does for every previous run on this Drive account.

In [ ]:
# --- Cell 3: download dataset ---
import kagglehub
dataset_path = kagglehub.dataset_download("andrewmvd/lung-and-colon-cancer-histopathological-images")
print("Dataset root:", dataset_path)

In [ ]:
# --- Cell 4: discover records ---
all_paths, all_indices = discover_records(dataset_path)
print(f"Discovered {len(all_paths)} images across {NUM_CLASSES} classes")
for idx in range(NUM_CLASSES):
    n = int((all_indices == idx).sum())
    print(f"  class {idx} ({INDEX_TO_NAME[idx]}): {n}")

In [ ]:
# --- Cell 5: load or build split ---
if SPLIT_PATH.exists():
    split = load_split(SPLIT_PATH)
    print(f"Loaded split from {SPLIT_PATH}")
else:
    split = stratified_split(all_paths, all_indices,
                             val_fraction=0.10, test_fraction=0.10, seed=SEED)
    save_split(split, SPLIT_PATH)
    print(f"Built + saved split at {SPLIT_PATH}")

train_paths, train_idx = split["train_paths"], split["train_indices"]
val_paths,   val_idx   = split["val_paths"],   split["val_indices"]
test_paths,  test_idx  = split["test_paths"],  split["test_indices"]
print(f"Train: {len(train_paths)}  Val: {len(val_paths)}  Test: {len(test_paths)}")

## 4 — Load precomputed DINOv2 features + BERT text encoder

The DINOv2 vision encoder lives entirely in the precompute notebook. Here we just open the HDF5 file and build a `{relative_path: 768-d feature}` dict, where the key is the POSIX relative path under the LC25000 `lung_colon_image_set/` root (e.g. `lung_image_sets/lung_n/lungn1.jpeg`). The training split's absolute paths are converted to the same key on the fly.

BERT loads as usual: `bert-base-uncased`, frozen.

In [ ]:
# --- Cell 6: load precomputed DINOv2 features + BERT text ---
def _load_dinov2_feature_cache(cache_path):
    """Load HDF5 cache -> (features (N,768) float32, list of rel_path keys).

    The HDF5 layout is the one written by DINOv2_feature_precompute.ipynb:
      features : (N, 768) float32 -- DINOv2 ViT-B/14 CLS token
      paths    : (N,) S512        -- POSIX relative path under the LC25000 root
    """
    with h5py.File(cache_path, "r") as f:
        feats = f["features"][:]
        keys  = [p.decode() if isinstance(p, bytes) else p for p in f["paths"][:]]
        attrs = {k: f.attrs[k] for k in f.attrs}
    assert feats.dtype == np.float32, feats.dtype
    assert feats.shape[1] == VISION_HIDDEN, feats.shape
    return feats, keys, attrs


def _load_bert_text(encoder_id=TEXT_ENCODER_ID):
    tokenizer = AutoTokenizer.from_pretrained(encoder_id)
    try:
        m = TFAutoModel.from_pretrained(encoder_id, from_pt=False)
    except (OSError, EnvironmentError, TypeError, ValueError):
        m = TFAutoModel.from_pretrained(encoder_id, from_pt=True)
    m.trainable = False
    return m, tokenizer, m.config.hidden_size


print(f"Loading precomputed DINOv2 features from {FEATURE_CACHE_PATH}...")
feat_arr, feat_keys, feat_attrs = _load_dinov2_feature_cache(FEATURE_CACHE_PATH)
PATH_TO_FEATURE = {k: i for i, k in enumerate(feat_keys)}
print(f"  features:     {feat_arr.shape}  ({feat_arr.dtype})")
print(f"  cache attrs:  { {k: feat_attrs[k] for k in feat_attrs} }")

print(f"\nLoading text encoder ({TEXT_ENCODER_ID})...")
text_backbone, tokenizer, text_hidden = _load_bert_text()
print(f"  text hidden_size = {text_hidden}")  # 768 for bert-base-uncased


def tokenize_prompts(tokenizer, prompts, max_len=MAX_LEN):
    out = tokenizer(prompts,
                    padding="max_length", truncation=True, max_length=max_len,
                    return_tensors="tf")
    return {k: tf.cast(v, tf.int32) for k, v in out.items()}


# Pre-tokenise the 5 class prompts (composed strategy).
class_prompts = [baseline_prompt(name) for name in CLASS_NAMES]
print("\nClass prompts:")
for p in class_prompts:
    print(f"  {p!r}")
class_token_dict = tokenize_prompts(tokenizer, class_prompts)
print(f"\nToken shapes: { {k: v.shape for k, v in class_token_dict.items()} }")

## 5 — Feature pipeline

No image decoding happens in this notebook -- each training/eval sample is a precomputed 768-d DINOv2 feature vector looked up by relative-path key. `build_train_dataset` materialises the (N, 768) feature tensor once and slices it via `tf.data.Dataset.from_tensor_slices`, returning `((feature_vector, token_dict), class_index)` per sample.

Path-to-key conversion: we strip the LC25000 dataset root prefix (`lung_colon_image_set/`) from each absolute path and POSIX-normalise the result, matching the keys written by the precompute notebook.

In [ ]:
# --- Cell 7: feature gather + dataset builder ---
LC25000_ROOT_TOKEN = "lung_colon_image_set"


def path_to_cache_key(abs_path):
    """Absolute LC25000 path -> POSIX relative key under lung_colon_image_set/.

    Mirrors the key scheme used by DINOv2_feature_precompute.ipynb.
    """
    p = Path(abs_path).as_posix()
    if f"/{LC25000_ROOT_TOKEN}/" not in p:
        raise ValueError(f"Path does not look like an LC25000 image: {abs_path!r}")
    return p.split(f"/{LC25000_ROOT_TOKEN}/", 1)[1]


def gather_features(abs_paths):
    """Return (N, 768) float32 feature array aligned with abs_paths."""
    feats = np.zeros((len(abs_paths), VISION_HIDDEN), dtype=np.float32)
    missing = []
    for i, p in enumerate(abs_paths):
        key = path_to_cache_key(p)
        idx = PATH_TO_FEATURE.get(key)
        if idx is None:
            missing.append(key)
            continue
        feats[i] = feat_arr[idx]
    if missing:
        sample = ", ".join(missing[:3])
        raise KeyError(
            f"{len(missing)} LC25000 paths missing from DINOv2 feature cache "
            f"(e.g. {sample}). Recompute the cache (delete "
            f"{FEATURE_CACHE_PATH} and re-run DINOv2_feature_precompute.ipynb)."
        )
    return feats


def build_train_dataset(paths, indices, batch_size, shuffle=True):
    """tf.data pipeline that yields ((feature, token_dict), class_idx).

    The feature tensor is pre-gathered once into a single (N, 768) array, so the
    per-step cost is just slicing + a dict lookup -- no image decoding.
    """
    n = len(paths)
    feats = gather_features(paths)
    idx_t = tf.constant(indices, dtype=tf.int32)
    feat_t = tf.constant(feats)
    token_lookup = {k: tf.constant(v) for k, v in class_token_dict.items()}

    def map_fn(feat, class_idx):
        token_dict = {k: v[class_idx] for k, v in token_lookup.items()}
        return (feat, token_dict), class_idx

    ds = tf.data.Dataset.from_tensor_slices((feat_t, idx_t))
    if shuffle:
        ds = ds.shuffle(buffer_size=min(n, 10_000), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(map_fn, num_parallel_calls=AUTOTUNE)
    ds = ds.batch(batch_size, drop_remainder=False)
    ds = ds.prefetch(AUTOTUNE)
    return ds


def build_eval_feature_dataset(paths, batch_size):
    feats = gather_features(paths)
    ds = tf.data.Dataset.from_tensor_slices(tf.constant(feats))
    ds = ds.batch(batch_size, drop_remainder=False)
    ds = ds.prefetch(AUTOTUNE)
    return ds


# Smoke-test: one sample
sample_ds = build_train_dataset(train_paths[:2], train_idx[:2], batch_size=2, shuffle=False)
for (feats, tokens), labels in sample_ds.take(1):
    print(f"features shape: {feats.shape}  dtype: {feats.dtype}")
    print(f"features range: [{tf.reduce_min(feats).numpy():.3f}, {tf.reduce_max(feats).numpy():.3f}]")
    print(f"labels: {labels.numpy().tolist()}")
    for k, v in tokens.items():
        print(f"  {k}: {v.shape} {v.dtype}")

## 6 — Model — `CLIPModel_DINOv2`

Projection head over precomputed DINOv2 features + BERT text encoder + `Dense(256, no bias) → LayerNorm` projection heads + learnable `logit_scale`. Symmetric InfoNCE loss in `_clip_loss`.

`encode_image` now takes a pre-gathered 768-d feature vector and applies only the projection head (no vision backbone forward pass at all). The trainable parameter count drops to **just the projection heads + temperature** -- the saved `weights.weights.h5` is ~400K parameters, same scale as exp_08's PLIP-backbone checkpoint, and the per-step cost is dominated by BERT.

In [ ]:
# --- Cell 8: CLIPModel_DINOv2 (projection-head only) ---
class CLIPModel_DINOv2(keras.Model):
    """Projection-head CLIP over precomputed DINOv2 features + frozen BERT.

    The vision backbone lives entirely in DINOv2_feature_precompute.ipynb; this
    Keras model receives a 768-d feature vector per image and trains only the
    projection heads + temperature.
    """

    def __init__(self, text_backbone, vision_hidden, text_hidden,
                 embed_dim=EMBED_DIM, init_temp=INIT_TEMP, **kwargs):
        super().__init__(**kwargs)
        self.text_backbone = text_backbone
        self.vision_hidden = vision_hidden
        self.text_hidden = text_hidden

        self.img_projection = keras.Sequential([
            keras.layers.Dense(embed_dim, use_bias=False, dtype="float32"),
            keras.layers.LayerNormalization(dtype="float32"),
        ], name="img_projection")
        self.text_projection = keras.Sequential([
            keras.layers.Dense(embed_dim, use_bias=False, dtype="float32"),
            keras.layers.LayerNormalization(dtype="float32"),
        ], name="text_projection")

        self.logit_scale = self.add_weight(
            name="logit_scale", shape=(),
            initializer=keras.initializers.Constant(math.log(1.0 / init_temp)),
            trainable=True, dtype="float32",
        )
        self.loss_tracker = keras.metrics.Mean(name="loss")

    @property
    def metrics(self):
        return [self.loss_tracker]

    def encode_image(self, vision_features, training=False):
        """vision_features: (B, 768) precomputed DINOv2 CLS embeddings."""
        x = self.img_projection(vision_features)
        return tf.math.l2_normalize(x, axis=-1)

    def encode_text(self, token_dict, training=False):
        kwargs = {"input_ids": token_dict["input_ids"],
                  "attention_mask": token_dict["attention_mask"]}
        if "token_type_ids" in token_dict:
            kwargs["token_type_ids"] = token_dict["token_type_ids"]
        out = self.text_backbone(**kwargs, training=False)
        pooled = getattr(out, "pooler_output", None)
        if pooled is None:
            pooled = out.last_hidden_state[:, 0, :]
        x = self.text_projection(pooled)
        return tf.math.l2_normalize(x, axis=-1)

    def _clip_loss(self, img_emb, txt_emb):
        img_emb = tf.cast(img_emb, tf.float32)
        txt_emb = tf.cast(txt_emb, tf.float32)
        logit_scale = tf.clip_by_value(self.logit_scale, -math.log(100.0), math.log(100.0))
        logits = tf.matmul(img_emb, txt_emb, transpose_b=True) * tf.exp(logit_scale)
        labels = tf.range(tf.shape(img_emb)[0], dtype=tf.int32)
        ce = tf.keras.losses.SparseCategoricalCrossentropy(
            from_logits=True, reduction=tf.keras.losses.Reduction.NONE,
        )
        loss_i = tf.reduce_mean(ce(labels, logits))
        loss_t = tf.reduce_mean(ce(labels, tf.transpose(logits)))
        return 0.5 * (loss_i + loss_t)

    def call(self, inputs, training=False):
        vision_features, token_dict = inputs
        return self.encode_image(vision_features, training=training), self.encode_text(token_dict, training=training)

    def train_step(self, data):
        (inputs, _y) = data
        with tf.GradientTape() as tape:
            img_emb, txt_emb = self(inputs, training=True)
            loss = self._clip_loss(img_emb, txt_emb)
        trainable = self.trainable_variables
        grads = tape.gradient(loss, trainable)
        self.optimizer.apply_gradients(zip(grads, trainable))
        self.loss_tracker.update_state(loss)
        return {"loss": self.loss_tracker.result()}

    def test_step(self, data):
        (inputs, _y) = data
        img_emb, txt_emb = self(inputs, training=False)
        loss = self._clip_loss(img_emb, txt_emb)
        self.loss_tracker.update_state(loss)
        return {"loss": self.loss_tracker.result()}


model = CLIPModel_DINOv2(text_backbone, VISION_HIDDEN, text_hidden)

# Warm-up forward pass.
dummy_feats = tf.zeros((1, VISION_HIDDEN), dtype=tf.float32)
dummy_toks = {k: tf.constant(v[:1]) for k, v in class_token_dict.items()}
_ = model((dummy_feats, dummy_toks), training=False)

optimizer = keras.optimizers.AdamW(
    learning_rate=LR, weight_decay=WD, global_clipnorm=GLOBAL_CLIPNORM,
)
model.compile(optimizer=optimizer)

n_trainable = sum(int(np.prod(v.shape)) for v in model.trainable_variables)
n_total     = sum(int(np.prod(v.shape)) for v in model.variables)
print(f"Trainable params: {n_trainable:,}  (~395K expected: img_proj 768*256 + text_proj 768*256 + 2*LN + 1)")
print(f"Total params:     {n_total:,}  (BERT ~110M + projections ~400K; DINOv2 lives in the precompute cache, not this model)")

## 7 — Train (or load cached weights)

Same protocol as exp_06 / exp_08: 30 epochs, EarlyStopping(patience=5, restore_best_weights), ReduceLROnPlateau(factor=0.5, patience=2). Weights saved to Drive at `<RUN_DIR>/weights.weights.h5`.

Expected runtime: much faster than exp_08 (no DINOv2 forward pass per step). The per-step cost is BERT + two projection heads. On a Colab T4 expect a few minutes per epoch instead of tens.

In [ ]:
# --- Cell 9: train ---
WEIGHTS_PATH = RUN_DIR / "weights.weights.h5"
HISTORY_PATH = RUN_DIR / "history.json"

if not FORCE_RETRAIN and WEIGHTS_PATH.exists():
    print(f"Found cached weights at {WEIGHTS_PATH}; loading.")
    model.load_weights(str(WEIGHTS_PATH))
    history = json.loads(HISTORY_PATH.read_text()) if HISTORY_PATH.exists() else None
else:
    print(f"No cached weights at {WEIGHTS_PATH}. Training from scratch.")
    train_ds = build_train_dataset(train_paths, train_idx, BATCH_SIZE, shuffle=True)
    val_ds   = build_train_dataset(val_paths, val_idx, BATCH_SIZE, shuffle=False)

    callbacks = [
        keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=EARLY_STOPPING_PATIENCE,
            restore_best_weights=True, verbose=1,
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss", factor=0.5, patience=2, verbose=1,
        ),
    ]

    history_obj = model.fit(
        train_ds, validation_data=val_ds, epochs=EPOCHS,
        callbacks=callbacks, verbose=2,
    )

    model.save_weights(str(WEIGHTS_PATH))
    history = {k: [float(v) for v in vs] for k, vs in history_obj.history.items()}
    HISTORY_PATH.write_text(json.dumps(history, indent=2))
    print(f"\nSaved weights -> {WEIGHTS_PATH}")
    print(f"Saved history -> {HISTORY_PATH}")

## 8 — In-distribution evaluation (LC25000 test)

In [ ]:
# --- Cell 10: classify test set ---
def encode_test_images(model, paths):
    ds = build_eval_feature_dataset(paths, BATCH_SIZE)
    out = []
    for batch in ds:
        emb = model.encode_image(batch, training=False)
        out.append(emb.numpy())
    return np.concatenate(out, axis=0)


print("Encoding LC25000 test images (via cached DINOv2 features)...")
test_image_embeddings = encode_test_images(model, test_paths)
print(f"  shape: {test_image_embeddings.shape}")

# Encode all 5 class prompts.
class_text_embeddings = model.encode_text(class_token_dict, training=False).numpy()
print(f"Class prompt embeddings: {class_text_embeddings.shape}")

similarity_scores = test_image_embeddings @ class_text_embeddings.T
predicted_class_ids = similarity_scores.argmax(axis=1).astype(np.int32)
true_class_ids = np.asarray(test_idx, dtype=np.int32)

acc      = accuracy_score(true_class_ids, predicted_class_ids)
bal_acc  = balanced_accuracy_score(true_class_ids, predicted_class_ids)
macro_f1 = f1_score(true_class_ids, predicted_class_ids, average="macro")
wgt_f1   = f1_score(true_class_ids, predicted_class_ids, average="weighted")

print("\nGlobal metrics:")
print(f"  accuracy          = {acc:.4f}")
print(f"  balanced_accuracy = {bal_acc:.4f}")
print(f"  macro_f1          = {macro_f1:.4f}")
print(f"  weighted_f1       = {wgt_f1:.4f}")

per_class = classification_report(
    true_class_ids, predicted_class_ids,
    labels=list(range(NUM_CLASSES)),
    target_names=[INDEX_TO_NAME[i] for i in range(NUM_CLASSES)],
    output_dict=True, zero_division=0,
)
per_class_df = pd.DataFrame(per_class).T
print("\nPer-class:")
print(per_class_df.round(4).to_string())

metrics_payload = {
    "experiment":          EXPERIMENT,
    "version":             VERSION,
    "image_encoder":       DINOV2_MODEL_ID,
    "text_encoder":        TEXT_ENCODER_ID,
    "prompt_strategy":     "composed",
    "preprocessing":       "ImageNet mean/std applied inside DINOv2_feature_precompute.ipynb (AutoImageProcessor)",
    "feature_cache":       str(FEATURE_CACHE_PATH),
    "accuracy":            float(acc),
    "balanced_accuracy":   float(bal_acc),
    "macro_f1":            float(macro_f1),
    "weighted_f1":         float(wgt_f1),
    "per_class":           per_class,
}
metrics_path = METRICS_DIR / f"{VERSION}_classification.json"
metrics_path.write_text(json.dumps(metrics_payload, indent=2))
print(f"\nSaved {metrics_path}")

## 9 — Confusion matrix

In [ ]:
# --- Cell 11: confusion matrix plot ---
cm_raw = confusion_matrix(true_class_ids, predicted_class_ids, labels=list(range(NUM_CLASSES)))
row_sums = cm_raw.sum(axis=1, keepdims=True)
cm_norm = np.divide(cm_raw, row_sums, where=row_sums > 0,
                    out=np.zeros_like(cm_raw, dtype=np.float64))

display_labels = [INDEX_TO_NAME[i] for i in range(NUM_CLASSES)]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, mat, title, fmt, cmap in [
    (axes[0], cm_raw,  "Confusion matrix (raw)",          "d",   "Blues"),
    (axes[1], cm_norm, "Confusion matrix (row-norm)",     ".2f", "Blues"),
]:
    im = ax.imshow(mat, cmap=cmap)
    ax.set_xticks(range(NUM_CLASSES)); ax.set_yticks(range(NUM_CLASSES))
    ax.set_xticklabels(display_labels, rotation=20, ha="right", fontsize=9)
    ax.set_yticklabels(display_labels, fontsize=9)
    ax.set_xlabel("predicted"); ax.set_ylabel("true")
    ax.set_title(title)
    for i in range(NUM_CLASSES):
        for j in range(NUM_CLASSES):
            ax.text(j, i, format(mat[i, j], fmt), ha="center", va="center",
                    color="white" if mat[i, j] > mat.max() * 0.6 else "black", fontsize=9)
    fig.colorbar(im, ax=ax, fraction=0.046)
plt.suptitle(f"LC25000 test CMs — {VERSION}", fontsize=11)
plt.tight_layout()
cm_path = PLOTS_DIR / f"{VERSION}_confusion_matrices.png"
fig.savefig(cm_path, dpi=140, bbox_inches="tight")
plt.show()
np.save(CM_DIR / f"{VERSION}_cm_raw.npy",  cm_raw)
np.save(CM_DIR / f"{VERSION}_cm_norm.npy", cm_norm)
print(f"Saved {cm_path}")

## 10 — Training history (loss curves)

In [ ]:
# --- Cell 12: plot training history ---
if history:
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(history["loss"], label="train loss")
    if "val_loss" in history:
        ax.plot(history["val_loss"], label="val loss")
    ax.set_xlabel("epoch"); ax.set_ylabel("loss")
    ax.set_title(f"{VERSION} — training history")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    loss_path = PLOTS_DIR / f"{VERSION}_loss.png"
    fig.savefig(loss_path, dpi=140, bbox_inches="tight")
    plt.show()
    print(f"Saved {loss_path}")
else:
    print("No history available (loaded from cache without history.json).")

## 11 — Shared-UMAP visualisation

Image embeddings + 5 class-prompt embeddings projected into a single UMAP — same convention used by the other CLIP-style notebooks. Class-prompt stars at the cluster centres make the per-class structure legible.

In [ ]:
# --- Cell 13: shared UMAP fit ---
combined = np.vstack([test_image_embeddings, class_text_embeddings])
reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, metric="cosine", random_state=SEED)
combined_2d = reducer.fit_transform(combined)
image_2d  = combined_2d[:len(test_image_embeddings)]
prompt_2d = combined_2d[len(test_image_embeddings):]

CLASS_COLOR = {
    0: "#2ca02c",  # benign lung
    1: "#ff7f0e",  # lung adeno
    2: "#9467bd",  # lung SCC
    3: "#08519c",  # benign colon
    4: "#a50f15",  # colon adeno
}

fig, ax = plt.subplots(figsize=(9, 7))
for cls in range(NUM_CLASSES):
    mask = (true_class_ids == cls)
    ax.scatter(image_2d[mask, 0], image_2d[mask, 1], s=8, alpha=0.55,
               c=CLASS_COLOR[cls], label=INDEX_TO_NAME[cls], edgecolors="none")
for cls in range(NUM_CLASSES):
    ax.scatter(prompt_2d[cls, 0], prompt_2d[cls, 1], s=460, marker="*",
               c=CLASS_COLOR[cls], edgecolors="white", linewidths=2.0, zorder=6)
    ax.annotate(f"prompt:\n{INDEX_TO_NAME[cls]}",
                (prompt_2d[cls, 0], prompt_2d[cls, 1]),
                xytext=(12, 10), textcoords="offset points",
                fontsize=8, fontweight="bold", zorder=7,
                bbox=dict(boxstyle="round,pad=0.25", facecolor="white",
                          edgecolor=CLASS_COLOR[cls], alpha=0.9, linewidth=1.5))
ax.legend(loc="best", fontsize=8, framealpha=0.85)
ax.set_xticks([]); ax.set_yticks([])
ax.set_title(f"{VERSION} — LC25000 test image embeddings + class prompts\nF1 (macro) = {macro_f1:.4f}")
plt.tight_layout()
umap_path = PLOTS_DIR / f"{VERSION}_umap.png"
fig.savefig(umap_path, dpi=140, bbox_inches="tight")
plt.show()
print(f"Saved {umap_path}")

## Summary

Run-time output:
- `<RUN_DIR>/weights.weights.h5` — trained projection-head weights (loadable by the OOD eval notebooks)
- `<RUN_DIR>/history.json` — per-epoch loss curves
- `results/metrics/exp_10_dinov2_backbone/dinov2_bert_composed_classification.json` — global + per-class in-dist metrics
- `results/plots/exp_10_dinov2_backbone/dinov2_bert_composed_{umap,loss,confusion_matrices}.png`
- `results/confusion_matrices/exp_10_dinov2_backbone/dinov2_bert_composed_cm_{raw,norm}.npy`

**Next step:** run the three OOD eval notebooks (NCT-CRC, Chaoyang, LungHist700) with `VARIANT = "dinov2_bert_composed"` to test whether the general-domain self-supervised ViT keeps PLIP's OOD advantage. If exp_10 OOD F1 ≈ exp_08 OOD F1, most of the advantage was the architecture / SSL family, not histopathology pretraining. If exp_10 OOD F1 lands closer to the exp_01 ResNet50 baseline, then the bottleneck really is histopathology pretraining.